# VehCutML — YOLO26s Segmentation
Upload `vehCut01.zip` to `MyDrive/VehCutML/vehCut01.zip`, select a T4 GPU, then run every cell in order. Upload/open this file as a notebook; do not paste its JSON into a code cell.

In [ ]:
!nvidia-smi
!pip -q install -U ultralytics pyyaml
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/VehCutML')
ZIP_PATH = DRIVE_ROOT / 'vehCut01.zip'
RUN_ROOT = DRIVE_ROOT / 'runs' / 'vehicel_detector'  # Name retained as requested.
WORK_ROOT = Path('/content/vehcut01_work')
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Upload vehCut01.zip to: {ZIP_PATH}')
if ZIP_PATH.stat().st_size < 1_000_000:
    raise RuntimeError(f'ZIP is too small: {ZIP_PATH.stat().st_size} bytes')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(WORK_ROOT, ignore_errors=True)
WORK_ROOT.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), str(WORK_ROOT))

# Supports a ZIP whose images/ and labels/ are at the root or below one wrapper directory.
candidates = [WORK_ROOT] + [p for p in WORK_ROOT.iterdir() if p.is_dir()]
SOURCE_ROOT = next((p for p in candidates if (p/'images'/'train').is_dir() and (p/'labels'/'train').is_dir()), None)
if SOURCE_ROOT is None:
    raise RuntimeError('Cannot find images/train and labels/train after extracting the ZIP.')
print('Source root:', SOURCE_ROOT)
print('Output run:', RUN_ROOT)

In [ ]:
# Validate image/label pairs and use the extracted data directly (no duplicate copy to Drive).
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
def stems(directory, extensions):
    return {p.stem for p in directory.iterdir() if p.is_file() and p.suffix.lower() in extensions}

for split in ('train', 'val'):
    image_dir, label_dir = SOURCE_ROOT/'images'/split, SOURCE_ROOT/'labels'/split
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise RuntimeError(f'Missing required split: {split}')
    image_stems, label_stems = stems(image_dir, IMAGE_EXTENSIONS), stems(label_dir, {'.txt'})
    missing_labels, missing_images = image_stems - label_stems, label_stems - image_stems
    print(f'{split}: paired={len(image_stems & label_stems)}, missing labels={len(missing_labels)}, missing images={len(missing_images)}')
    if missing_labels or missing_images:
        raise RuntimeError(f'Unpaired image/label data in {split}. Stop before training.')

data_yaml = WORK_ROOT / 'data.yaml'
data_yaml.write_text(f'''path: {SOURCE_ROOT}
train: images/train
val: images/val
nc: 2
names:
  0: BSD
  1: BSV
''', encoding='utf-8')
print(data_yaml.read_text())

The labels are polygons, so this trains a license-plate segmentation model with `yolo26s-seg.pt`. It is separate from the vehicle detector.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26s-seg.pt')
model.train(
    data=str(data_yaml), epochs=80, imgsz=640, batch=8, device=0,
    optimizer='AdamW', lr0=0.0005, patience=20,
    project='/content/runs', name='vehicel_detector',
    save=True, save_period=10, pretrained=True
)

In [ ]:
SOURCE_RUN = Path('/content/runs/vehicel_detector')
BEST_WEIGHT = SOURCE_RUN / 'weights' / 'best.pt'
if not BEST_WEIGHT.is_file():
    raise RuntimeError(f'best.pt was not produced: {BEST_WEIGHT}')
shutil.copytree(SOURCE_RUN, RUN_ROOT, dirs_exist_ok=True)
print('Saved best.pt, last.pt, results.csv, results.png and args.yaml to:', RUN_ROOT)